In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

# 1. Database Connection Credentials
DB_USER = "root"
DB_PASSWORD = "admin123"  # 👈 Replace with your MySQL password
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "retail_loan_risk"  # 👈 Replace with your exact schema name

# Encode password to handle special characters safely
encoded_password = urllib.parse.quote_plus(DB_PASSWORD)

# 2. Create the SQLAlchemy Engine
connection_string = f"mysql+mysqlconnector://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    # Test connection
    with engine.connect() as conn:
        print("✅ Successfully connected to the MySQL Database!")
except Exception as e:
    print(f"❌ Connection failed: {e}")


✅ Successfully connected to the MySQL Database!


In [2]:
# 3. Pull Matured Loans with Credit History for PD Modeling
# We join loans and credit_histories, filtering for matured outcomes (Fully Paid vs. Charged Off)
pd_query = """
SELECT 
    l.loan_id,
    l.borrower_id,
    l.loan_amnt,
    l.term_months,
    l.int_rate,
    l.installment,
    l.grade,
    l.sub_grade,
    ch.dti,
    ch.open_acc,
    ch.pub_rec,
    ch.revol_bal,
    ch.revol_util,
    ch.total_acc,
    snap.loan_status  -- This contains our target labels
FROM loans l
INNER JOIN credit_histories ch ON l.borrower_id = ch.borrower_id
INNER JOIN loan_perf_snapshots snap ON l.loan_id = snap.loan_id
WHERE snap.loan_status IN ('Fully Paid', 'Charged Off'); -- We model on resolved loans
"""

df_pd_modeling = pd.read_sql(pd_query, con=engine)
print(f"Dataset Loaded! Shape: {df_pd_modeling.shape}")
print(df_pd_modeling.head())


Dataset Loaded! Shape: (1303607, 15)
  loan_id  borrower_id  loan_amnt  term_months  int_rate  installment grade  \
0     101          101    30000.0           36     22.35      1151.16     D   
1     153          153    40000.0           60     16.14       975.71     C   
2     171          171    20000.0           36      7.56       622.68     A   
3     187          187     4500.0           36     11.31       147.99     B   
4     216          216     8425.0           36     27.27       345.18     E   

  sub_grade    dti  open_acc  pub_rec  revol_bal  revol_util  total_acc  \
0        D5  30.46        11        1    15603.0        37.0         19   
1        C4  50.53        18        0    34971.0        64.5         37   
2        A3  18.92         9        0    25416.0        29.9         19   
3        B3   4.64        12        0     4472.0        15.3         25   
4        E5  12.37        21        0    36812.0        65.7         37   

  loan_status  
0  Fully Paid  
1  Fu

In [4]:
# 4. FEATURE ENGINEERING PIPELINE
print("\n" + "="*50)
print("⚙️ STARTING FEATURE ENGINEERING")
print("="*50)

# Copy dataframe to keep raw data pristine
df_features = df_pd_modeling.copy()

# A. Create the Binary Target Column (default_flag)
# 1 = Default (Charged Off), 0 = No Default (Fully Paid)
df_features['default_flag'] = df_features['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)
print(f"✅ Created target 'default_flag'. Mean default rate: {df_features['default_flag'].mean()*100:.2f}%")

# B. Process 'term_months' to be numeric (if it contains text like '36 months')
if df_features['term_months'].dtype == 'object':
    df_features['term_months'] = df_features['term_months'].str.extract('(\\d+)').astype(float)
    print("✅ Parsed 'term_months' into numeric values.")

# C. Handle Missing Values (Imputation)
# If 'revol_util' has missing percentages, fill them with the median value (robust to outliers)
if df_features['revol_util'].isnull().sum() > 0:
    median_util = df_features['revol_util'].median()
    df_features['revol_util'].fillna(median_util, inplace=True)
    print(f"✅ Imputed missing 'revol_util' values with median ({median_util:.2f}).")

# D. Encode Ordinal/Categorical Columns (loan grade)
# Standard credit risk practice: convert alphanumeric grades (A-G) to numeric ranks (1-7)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df_features['grade_numeric'] = df_features['grade'].map(grade_map)
print("✅ Encoded loan 'grade' to 'grade_numeric'.")

# Drop unnecessary columns that won't be fed into model training
columns_to_drop = ['loan_id', 'borrower_id', 'loan_status', 'grade', 'sub_grade']
df_model_input = df_features.drop(columns=columns_to_drop)

print(f"✅ Feature Engineering complete! Final training dataset columns: \n{list(df_model_input.columns)}")



⚙️ STARTING FEATURE ENGINEERING
✅ Created target 'default_flag'. Mean default rate: 20.07%
✅ Encoded loan 'grade' to 'grade_numeric'.
✅ Feature Engineering complete! Final training dataset columns: 
['loan_amnt', 'term_months', 'int_rate', 'installment', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'default_flag', 'grade_numeric']


In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, 
    classification_report, 
    confusion_matrix
)

# =====================================================================
# 5. DATA SPLITTING & PREPROCESSING
# =====================================================================
print("\n" + "="*50)
print("📊 SPLITTING & SCALING DATA FOR PD MODELING")
print("="*50)

# Drop any remaining rows with unexpected nulls in other numeric columns
df_clean = df_model_input.dropna().copy()

# Separate features (X) and target (y)
X = df_clean.drop(columns=['default_flag'])
y = df_clean['default_flag']

# Stratified Train-Test Split (80% Training, 20% Testing)
# 'stratify=y' ensures both sets have the exact same percentage of defaults
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"• Training Set: {X_train.shape[0]:,} loans (Default Rate: {y_train.mean()*100:.2f}%)")
print(f"• Testing Set:  {X_test.shape[0]:,} loans (Default Rate: {y_test.mean()*100:.2f}%)")

# Standardize Features (Crucial for Logistic Regression convergence and coefficient comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ⚠️ Only transform on test to prevent data leakage!

# =====================================================================
# 6. TRAIN LOGISTIC REGRESSION PD MODEL
# =====================================================================
print("\n⏳ Training Logistic Regression PD Model...")

# Initialize and train the model
pd_model = LogisticRegression(max_iter=1000, random_state=42)
pd_model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")

# =====================================================================
# 7. EXTRACT PROBABILITIES (PD) & EVALUATE PERFORMANCE
# =====================================================================
# Predict continuous Probability of Default (PD) — Column 1 is probability of class 1 (Default)
y_pred_pd = pd_model.predict_proba(X_test_scaled)[:, 1]

# Predict binary class for confusion matrix (using a 0.50 cutoff or custom threshold)
y_pred_class = pd_model.predict(X_test_scaled)

# Calculate Credit Risk Industry Metrics: ROC-AUC and Gini Coefficient
roc_auc = roc_auc_score(y_test, y_pred_pd)
gini_score = (2 * roc_auc) - 1

print("\n" + "="*50)
print("📈 PD MODEL PERFORMANCE METRICS")
print("="*50)
print(f"• ROC-AUC Score:    {roc_auc:.4f} (Benchmark: > 0.70 is good for retail credit)")
print(f"• Gini Coefficient: {gini_score:.4f} (Benchmark: > 0.40 is strong in banking)")
print(f"• Average Predicted PD on Test Set: {y_pred_pd.mean()*100:.2f}%")
print(f"• Actual Default Rate on Test Set:  {y_test.mean()*100:.2f}%")

# =====================================================================
# 8. FEATURE IMPORTANCE (ODDS RATIOS & COEFFICIENTS)
# =====================================================================
# Extract coefficients to explain what drives default risk
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (Log-Odds)': pd_model.coef_[0],
    'Odds_Ratio': np.exp(pd_model.coef_[0])
}).sort_values(by='Coefficient (Log-Odds)', ascending=False)

print("\n🔍 Risk Drivers (Top Positive = Increases Default Risk):")
print(feature_importance.to_string(index=False))

# Attach predicted PDs back to a sample of the test set to inspect
df_results_preview = X_test.copy()
df_results_preview['Actual_Default'] = y_test
df_results_preview['Predicted_PD'] = np.round(y_pred_pd, 4)

print("\n🔍 Sample Borrower PD Predictions:")
print(df_results_preview[['loan_amnt', 'int_rate', 'dti', 'grade_numeric', 'Actual_Default', 'Predicted_PD']].head(10))



📊 SPLITTING & SCALING DATA FOR PD MODELING
• Training Set: 1,042,885 loans (Default Rate: 20.07%)
• Testing Set:  260,722 loans (Default Rate: 20.07%)

⏳ Training Logistic Regression PD Model...
✅ Model training complete!

📈 PD MODEL PERFORMANCE METRICS
• ROC-AUC Score:    0.6970 (Benchmark: > 0.70 is good for retail credit)
• Gini Coefficient: 0.3940 (Benchmark: > 0.40 is strong in banking)
• Average Predicted PD on Test Set: 20.09%
• Actual Default Rate on Test Set:  20.07%

🔍 Risk Drivers (Top Positive = Increases Default Risk):
      Feature  Coefficient (Log-Odds)  Odds_Ratio
grade_numeric                0.340772    1.406033
  term_months                0.222934    1.249738
          dti                0.167666    1.182541
     int_rate                0.159254    1.172635
     open_acc                0.156737    1.169688
  installment                0.138329    1.148353
      pub_rec                0.047190    1.048321
   revol_util                0.042799    1.043728
    loan_am

In [6]:
# =====================================================================
# 9. INTEGRATING PD, LGD, & EAD FOR ECL COMPUTATION
# =====================================================================
print("\n" + "="*50)
print("💰 COMPUTING PORTFOLIO EXPECTED CREDIT LOSS (ECL)")
print("="*50)

# Create a copy of the results dataframe for the ECL simulation
df_ecl = df_results_preview.copy()

# A. EXPOSURE AT DEFAULT (EAD)
# For this matured snapshot, what is left to lose at default is the unpaid principal balance.
# Let's approximate the EAD for our simulation as: original loan amount minus paid principal.
# (For simplicty in this model, we'll use outstanding amount or funded amount)
df_ecl['EAD'] = df_ecl['loan_amnt']

# B. LOSS GIVEN DEFAULT (LGD)
# Based on our previous SQL recovery analysis, let's assume a standard LGD profile by Loan Grade:
# Better grades have higher recoveries (lower LGD), worse grades have lower recoveries (higher LGD).
lgd_by_grade = {
    1: 0.65,  # Grade A: 65% LGD (35% recovery rate)
    2: 0.70,  # Grade B: 70% LGD (30% recovery rate)
    3: 0.75,  # Grade C: 75% LGD (25% recovery rate)
    4: 0.80,  # Grade D: 80% LGD (20% recovery rate)
    5: 0.82,  # Grade E: 82% LGD (18% recovery rate)
    6: 0.85,  # Grade F: 85% LGD (15% recovery rate)
    7: 0.90   # Grade G: 90% LGD (10% recovery rate)
}
df_ecl['LGD'] = df_ecl['grade_numeric'].map(lgd_by_grade)

# C. PROBABILITY OF DEFAULT (PD)
# Use our model's predicted probabilities
df_ecl['PD'] = df_ecl['Predicted_PD']

# D. EXPECTED CREDIT LOSS (ECL) CALCULATION
# ECL = PD * LGD * EAD
df_ecl['ECL_Amount'] = df_ecl['PD'] * df_ecl['LGD'] * df_ecl['EAD']

# =====================================================================
# 10. PORTFOLIO-LEVEL ECL AGGREGATIONS
# =====================================================================
print("\n📊 Total Test Portfolio Summary:")
total_exposure = df_ecl['EAD'].sum()
total_ecl = df_ecl['ECL_Amount'].sum()
portfolio_ecl_pct = (total_ecl / total_exposure) * 100

print(f"• Total Portfolio Outstanding Exposure (EAD): ${total_exposure:,.2f}")
print(f"• Total Expected Credit Loss (ECL) Buffer:    ${total_ecl:,.2f}")
print(f"• Portfolio-Level ECL Rate:                     {portfolio_ecl_pct:.2f}%")

# Aggregate ECL metrics by Loan Grade
ecl_summary_by_grade = df_ecl.groupby('grade_numeric').agg(
    loan_count=('PD', 'count'),
    total_exposure=('EAD', 'sum'),
    avg_pd=('PD', 'mean'),
    avg_lgd=('LGD', 'mean'),
    total_ecl=('ECL_Amount', 'sum')
).reset_index()

ecl_summary_by_grade['ecl_pct_of_exposure'] = (ecl_summary_by_grade['total_ecl'] / ecl_summary_by_grade['total_exposure']) * 100

# Map numeric grade back to letters for reporting
reverse_grade_map = {v: k for k, v in grade_map.items()}
ecl_summary_by_grade['Grade'] = ecl_summary_by_grade['grade_numeric'].map(reverse_grade_map)

print("\n🔍 ECL Risk Breakdown by Loan Grade:")
columns_to_show = ['Grade', 'loan_count', 'total_exposure', 'avg_pd', 'avg_lgd', 'total_ecl', 'ecl_pct_of_exposure']
print(ecl_summary_by_grade[columns_to_show].to_string(index=False, formatters={
    'total_exposure': lambda x: f"${x:,.2f}",
    'total_ecl': lambda x: f"${x:,.2f}",
    'avg_pd': lambda x: f"{x*100:.2f}%",
    'avg_lgd': lambda x: f"{x*100:.2f}%",
    'ecl_pct_of_exposure': lambda x: f"{x:.2f}%"
}))



💰 COMPUTING PORTFOLIO EXPECTED CREDIT LOSS (ECL)

📊 Total Test Portfolio Summary:
• Total Portfolio Outstanding Exposure (EAD): $3,768,654,200.00
• Total Expected Credit Loss (ECL) Buffer:    $629,294,975.15
• Portfolio-Level ECL Rate:                     16.70%

🔍 ECL Risk Breakdown by Loan Grade:
Grade  loan_count    total_exposure avg_pd avg_lgd       total_ecl ecl_pct_of_exposure
    A       45193   $628,419,325.00  8.56%  65.00%  $34,921,179.63               5.56%
    B       76121 $1,006,586,850.00 13.44%  70.00%  $96,621,707.12               9.60%
    C       73718 $1,050,774,300.00 20.36%  75.00% $166,088,391.87              15.81%
    D       39014   $595,872,500.00 29.09%  80.00% $143,876,268.50              24.15%
    E       18507   $327,095,500.00 40.43%  82.00% $111,352,012.84              34.04%
    F        6370   $122,386,450.00 52.28%  85.00%  $55,305,110.17              45.19%
    G        1799    $37,519,275.00 62.15%  90.00%  $21,130,305.01              56.32%


In [7]:
# Export 1: Loan-level test predictions & ECL for Page 1 & Calibration
df_ecl['Grade'] = df_ecl['grade_numeric'].map(reverse_grade_map)
df_ecl.to_csv('powerbi_ecl_loan_predictions.csv', index=False)

# Export 2: Feature Importance table for Page 2 Tornado Chart
feature_importance.to_csv('powerbi_pd_feature_importance.csv', index=False)

print("✅ Power BI data files exported successfully!")


✅ Power BI data files exported successfully!
